In [6]:
# hilangkan warning tqdm
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

In [7]:
load_dotenv()

True

In [8]:
# Define the persistent directory for Chroma database
persistent_directory = "db/chroma_db"

# Load embeddings and vector store
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma(
    persist_directory=persistent_directory,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"},
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3886.62it/s]


In [9]:
# Search for relevant documents
query = "How much did Microsoft pay to acquire GitHub?"

retriever = db.as_retriever(search_kwargs={"k": 5})

# retriever = db.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={
#         "k": 5,
#         "score_threshold": 0.3  # Only return chunks with cosine similarity ≥ 0.3
#     }
# )
relevant_docs = retriever.invoke(query)


In [10]:
print(f"User Query: {query}")
# Display results
print("--- Context ---")
for i, doc in enumerate(relevant_docs, 1):
    print(f"Document {i}:\n{doc.page_content}\n")

User Query: How much did Microsoft pay to acquire GitHub?
--- Context ---
Document 1:
117. "Microsoft's 2018, part 1: Open source, wobbly Windows and everyone's going to the cloud"
(https://www.theregister.co.uk/2018/12/25/microsoft_year_in_review_2018/). The Register.
Archived (https://web.archive.org/web/20190103060059/https://www.theregister.co.uk/2018/
12/25/microsoft_year_in_review_2018/) from the original on January 3, 2019. Retrieved
January 3, 2019.

118. "Microsoft to acquire GitHub for $7.5 billion" (https://news.microsoft.com/2018/06/04/microso
ft-to-acquire-github-for-7-5-billion/). Microsoft. June 4, 2018. Archived (https://web.archive.or
g/web/20180604142244/https://news.microsoft.com/2018/06/04/microsoft-to-acquire-github-
for-7-5-billion/) from the original on June 4, 2018.

Document 2:
119. "Microsoft completes GitHub acquisition" (https://web.archive.org/web/20190112212059/http
s://www.msn.com/en-us/news/technology/microsoft-completes-github-acquisition/ar-BBOVV
OT). 

In [12]:
# Combine the query and the relevant document contents
combined_input = f"""Based on the following documents, please answer this question: {query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in relevant_docs])}
"""

# Please provide a clear, helpful answer using only the information from these documents. If you can't find the answer in the documents, say "I don't have enough information to answer that question based on the provided documents.

# Create a ChatGoogleGenerativeAI model
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

# Define the messages for the model
messages = [
    SystemMessage(content="""You are a helpful assistant. 
    Answer the user's question STRICTLY and ONLY based on the provided documents.

    Rules:
    1. If the answer is found in the documents → answer directly and concisely.
    2. If the answer is NOT found in the documents → respond ONLY with: "I don't have enough information to answer that question based on the provided documents." Do NOT add any extra explanation.
    3. Do NOT use any knowledge outside of the provided documents.
    4. Do NOT speculate or infer beyond what is explicitly stated."""),
    HumanMessage(content=combined_input),
]

# Invoke the model with the combined input
result = model.invoke(messages)

In [13]:
# Display the full result and content only
print("\n--- Generated Response ---")
# print("Full result:")
# print(result)
print("Content only:")
print(result.content)


--- Generated Response ---
Content only:
Microsoft officially announced the acquisition of GitHub for $7.5 billion.
